# RAG Colab: FAISS + BM25 Local + Qwen2.5-3B


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 1) Install dependencies
!pip -q uninstall -y langchain-classic langgraph-prebuilt langgraph
!pip -q install -U \
  "pandas==2.2.2" \
  "numpy<2.0" \
  "scikit-learn==1.6.1" \
  "openpyxl==3.1.5" \
  "tqdm==4.67.1" \
  "rank-bm25==0.2.2" \
  "faiss-cpu==1.8.0.post1" \
  "sentence-transformers==3.0.1" \
  "transformers==4.53.3" \
  "accelerate==1.9.0" \
  "langchain-core==0.3.84" \
  "langchain-community==0.3.31" \
  "langchain-text-splitters==0.3.11"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.1/367.1 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 459.1/459.1 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Run this cell to kill the process, which will allow you to restart the runtime and use the newly installed dependencies.
# After run this cell, please run the next cell to continue with the rest of the code.
# Just run the next cell, dont run the above cell again
import os
os.kill(os.getpid(), 9)

In [6]:
import faiss, requests, numpy
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
print("OK")

OK


In [7]:
from huggingface_hub import model_info
model_info("Qwen/Qwen2.5-3B-Instruct")
print("OK")

OK


In [ ]:
# 2) Config
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/LLM_RAG_PROJECT_BASE')

FAISS_INDEX_DIR = ROOT / "data/faiss/law_chunks_e5_base"
CHUNKS_JSONL = ROOT / "data/datahuggingface/corpus_important_docs_chunks.jsonl"
EVAL_CSV = ROOT / "data/evaluate/evaluate.csv"

EVAL_LIMIT = 100
RUN_NAME = f"eval150_fixed_base_colab_local_bm25"
OUTPUT_DIR = ROOT / "analysis/eval_runs"

# retrieval
BM25_K = 30
FAISS_K = 30
RRF_K = 60
FINAL_K = 5
MAX_CHUNKS_PER_DOC = 2
MIN_QUERY_TERM_OVERLAP = 2
KEEP_AT_LEAST = 5
CONTEXT_NEIGHBOR_WINDOW = 1
CONTEXT_SAME_ARTICLE_LIMIT = 2
CONTEXT_MAX_ADDITIONAL = 4
MAX_CONTEXT_TOKENS = 2200
DOC_FOCUS_MODE = "auto"  # off / auto / strict
DOC_FOCUS_MAX_PRIMARY_DOCS = 1
DOC_FOCUS_PRIMARY_MARGIN = 0.12
DOC_FOCUS_PRIMARY_DOC_QUOTA = 4
DOC_FOCUS_OTHER_DOC_QUOTA = 1

# models
E5_MODEL = "intfloat/multilingual-e5-base"
METRIC_EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
DEVICE = "cuda"
PREFIX_MODE = "query_passage"
NORMALIZE = True

USE_RERANKER = True
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
RERANKER_DEVICE = "cuda"
RERANKER_TOP_N = 20

# LLM setup
# local: tai thang model tu HuggingFace tren Colab
# api: goi server ngoai (Ollama/vLLM/OpenAI-compatible)
LLM_BACKEND = "local"  # "local" hoac "api"
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
LLM_LOCAL_DEVICE = "cuda"
LLM_LOCAL_DTYPE = "auto"  # auto/float16/bfloat16/float32
LLM_TRUST_REMOTE_CODE = False

# chi dung khi LLM_BACKEND="api"
LLM_BASE_URL = "http://localhost:11434"
TEMPERATURE = 0.0
MAX_TOKENS = 768
LLM_TIMEOUT = 900

for p in [FAISS_INDEX_DIR, CHUNKS_JSONL, EVAL_CSV]:
    print(p, "OK" if p.exists() else "MISSING")


/content/drive/MyDrive/LLM_RAG_PROJECT_SECOND/data/faiss/law_chunks_e5_base OK
/content/drive/MyDrive/LLM_RAG_PROJECT_SECOND/data/datahuggingface/corpus_important_docs_chunks.jsonl OK
/content/drive/MyDrive/LLM_RAG_PROJECT_SECOND/data/evaluate/evaluate.csv OK


In [9]:
!pip -q install opensearch-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.7/385.7 kB 10.7 MB/s eta 0:00:00


In [10]:
# 3) Imports from current pipeline scripts
import json
import re
import sys
from pathlib import Path
from typing import Any

import importlib
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from langchain_community.vectorstores import FAISS
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

SCRIPTS_DIR = ROOT / "scripts"
(SCRIPTS_DIR / "__init__.py").touch(exist_ok=True)
if str(ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(ROOT.resolve()))
importlib.invalidate_caches()

from scripts.rag_hybrid_opensearch_faiss_qwen import (
    apply_document_focus,
    E5Embeddings,
    RetrievedItem,
    build_answer_prompt,
    build_context,
    expand_context_chunks,
    enrich_and_filter_by_relevance,
    extract_target_article_numbers,
    extract_target_doc_numbers,
    extract_query_terms,
    generate_llm_answer,
    limit_chunks_per_doc,
    load_chunk_store,
    rerank_chunks,
    rrf_fuse,
)

from scripts.evaluate_rag_random_subset import (
    fix_mojibake,
    fold_text,
    normalize_text,
    jaccard_similarity,
    token_overlap_score,
    bleu_score_simple,
    rouge_l_score,
)


In [11]:
import scripts.rag_hybrid_opensearch_faiss_qwen as rag

rag.SYSTEM_PROMPT = (
    "Bạn là trợ lý pháp lý Việt Nam. "
    "Chỉ được sử dụng thông tin trong CONTEXT, không dùng kiến thức bên ngoài. "
    "Mục tiêu là TRÍCH XUẤT ĐẦY ĐỦ quy định liên quan, không trả lời tóm tắt thiếu ý. "
    "Nếu câu hỏi nêu rõ số hiệu văn bản (ví dụ 02/2025/TT-NHNN), phải ưu tiên văn bản đó làm căn cứ chính. "
    "Chỉ dùng văn bản khác khi CONTEXT thể hiện rõ là văn bản liên quan trực tiếp (sửa đổi, dẫn chiếu, điều kiện áp dụng). "
    "Nếu điều khoản có danh sách điểm/khoản (a, b, c... hoặc 1, 2, 3...), phải liệt kê đủ các ý liên quan đến câu hỏi; không được bỏ sót. "
    "Không tự thêm nội dung không có trong CONTEXT. "
    "Không dùng tiếng Anh trong tiêu đề/nội dung trả lời. "
    "Nếu dữ liệu thiếu, nêu rõ phần thiếu và viết đúng câu: 'Không đủ dữ liệu trong CONTEXT'. "
    "Luôn gắn trích dẫn nguồn dạng [1], [2], ... cho từng ý quan trọng."
)

rag.ANSWER_REQUIREMENTS = (
    "Yêu cầu trả lời:\n"
    "1) Mở đầu bằng câu 'Căn cứ ...' và nêu đúng văn bản + điều/khoản/điểm chính.\n"
    "2) Trả lời đúng trọng tâm câu hỏi, theo hướng trích xuất quy định:\n"
    "   - Nếu hỏi 'trường hợp nào/điều kiện nào/nghĩa vụ gì' => liệt kê đầy đủ từng trường hợp/điều kiện/nghĩa vụ.\n"
    "   - Nếu hỏi kèm 'quy định/thủ tục như thế nào' => nêu thêm phần thủ tục, trình tự, lưu ý áp dụng (nếu có trong CONTEXT).\n"
    "3) Giữ sát câu chữ pháp lý trong CONTEXT; không diễn giải suy đoán.\n"
    "4) Nếu có ngoại lệ/giới hạn/phạm vi áp dụng trong điều khoản thì phải nêu đủ.\n"
    "5) Kết thúc bằng 'Tóm lại:' với kết luận ngắn gọn, đúng với các ý đã liệt kê ở trên.\n"
    "6) Không được kết thúc khi chưa liệt kê đủ các ý liên quan trong điều/khoản chính.\n"
    "7) Không dùng cách ghi sai định dạng như 'Điều 12.0' hoặc 'Khoản 1.0'; ghi đúng 'Điều 12', 'khoản 1'.\n"
    "8) Mỗi ý quan trọng phải có trích dẫn [n]. Nếu không đủ dữ liệu thì ghi: 'Không đủ dữ liệu trong CONTEXT'.\n"
)


In [12]:

# 4) Build local BM25 index + load FAISS
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]{2,}", fold_text(text))

def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

chunk_rows = read_jsonl(CHUNKS_JSONL)
bm25_corpus_tokens = [tokenize(str(r.get("chunk_text") or "")) for r in chunk_rows]
bm25 = BM25Okapi(bm25_corpus_tokens)

emb = E5Embeddings(
    model_name=E5_MODEL,
    device=DEVICE,
    normalize=NORMALIZE,
    prefix_mode=PREFIX_MODE,
)
metric_emb = E5Embeddings(
    model_name=METRIC_EMBEDDING_MODEL,
    device=DEVICE,
    normalize=True,
    prefix_mode=PREFIX_MODE,
)
faiss_vs = FAISS.load_local(str(FAISS_INDEX_DIR), emb, allow_dangerous_deserialization=True)
chunk_store = load_chunk_store(str(CHUNKS_JSONL.resolve()))

print("chunks:", len(chunk_rows))


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

chunks: 34374


In [13]:

# 5) Retriever with BM25 local + FAISS
def search_bm25_local(query: str, top_k: int) -> list[RetrievedItem]:
    q_tokens = tokenize(query)
    scores = bm25.get_scores(q_tokens)
    if len(scores) == 0:
        return []

    idxs = np.argsort(scores)[::-1][:top_k]
    out: list[RetrievedItem] = []
    rank = 1
    for i in idxs:
        s = float(scores[i])
        if s <= 0:
            continue
        row = chunk_rows[int(i)]
        cid = str(row.get("chunk_id") or "")
        if not cid:
            continue
        out.append(
            RetrievedItem(
                chunk_id=cid,
                payload=row,
                source="bm25_local",
                rank=rank,
                raw_score=s,
            )
        )
        rank += 1
    return out

def search_faiss_loaded(query: str, top_k: int) -> list[RetrievedItem]:
    hits = faiss_vs.similarity_search_with_score(query, k=top_k)
    out: list[RetrievedItem] = []
    for rank, (doc, score) in enumerate(hits, start=1):
        meta = doc.metadata or {}
        payload = dict(meta)
        payload.setdefault("chunk_text", doc.page_content)
        cid = str(payload.get("chunk_id") or "")
        if not cid:
            continue
        out.append(
            RetrievedItem(
                chunk_id=cid,
                payload=payload,
                source="faiss",
                rank=rank,
                raw_score=float(score),
            )
        )
    return out

def rag_answer_for_question(question: str) -> tuple[str, list[dict[str, Any]]]:
    bm25_hits = search_bm25_local(question, BM25_K)
    faiss_hits = search_faiss_loaded(question, FAISS_K)

    target_docs = extract_target_doc_numbers(question)
    target_articles = extract_target_article_numbers(question)

    fused = rrf_fuse([bm25_hits, faiss_hits], k=RRF_K)
    query_terms = extract_query_terms(question)
    fused = enrich_and_filter_by_relevance(
        fused,
        query_terms=query_terms,
        min_overlap=MIN_QUERY_TERM_OVERLAP,
        keep_at_least=KEEP_AT_LEAST,
        required_doc_numbers=target_docs,
        required_article_numbers=target_articles,
    )
    fused = rerank_chunks(
        query=question,
        chunks=fused,
        use_reranker=USE_RERANKER,
        reranker_model=RERANKER_MODEL,
        reranker_device=RERANKER_DEVICE,
        reranker_top_n=RERANKER_TOP_N,
    )
    fused = apply_document_focus(
        chunks=fused,
        mode=DOC_FOCUS_MODE,
        required_doc_numbers=target_docs,
        max_primary_docs=DOC_FOCUS_MAX_PRIMARY_DOCS,
        primary_score_margin=DOC_FOCUS_PRIMARY_MARGIN,
        primary_doc_quota=DOC_FOCUS_PRIMARY_DOC_QUOTA,
        other_doc_quota=DOC_FOCUS_OTHER_DOC_QUOTA,
    )
    fused = limit_chunks_per_doc(fused, max_chunks_per_doc=MAX_CHUNKS_PER_DOC)
    base_chunks = fused[:FINAL_K]
    if not base_chunks:
        return "", []

    final_chunks = expand_context_chunks(
        base_chunks=base_chunks,
        store=chunk_store,
        neighbor_window=CONTEXT_NEIGHBOR_WINDOW,
        same_article_limit=CONTEXT_SAME_ARTICLE_LIMIT,
        max_additional=CONTEXT_MAX_ADDITIONAL,
    )
    context = build_context(final_chunks, max_context_tokens=MAX_CONTEXT_TOKENS)
    prompt = build_answer_prompt(question, context)
    answer = generate_llm_answer(
        llm_backend=LLM_BACKEND,
        llm_model=LLM_MODEL,
        user_prompt=prompt,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        llm_timeout=LLM_TIMEOUT,
        llm_base_url=LLM_BASE_URL,
        llm_local_device=LLM_LOCAL_DEVICE,
        llm_local_dtype=LLM_LOCAL_DTYPE,
        llm_trust_remote_code=LLM_TRUST_REMOTE_CODE,
    )
    return answer, final_chunks


In [14]:
print("LLM_BACKEND =", LLM_BACKEND)
print("LLM_MODEL   =", LLM_MODEL)

LLM_BACKEND = local
LLM_MODEL   = Qwen/Qwen2.5-3B-Instruct


In [15]:
# 6) Evaluate fixed evaluate.csv (same metrics, no random subset)
def cosine_similarity_score(pred: str, true: str) -> float:
    if not pred or not true:
        return 0.0
    pv = metric_emb.embed_query(pred)
    tv = metric_emb.embed_query(true)
    return float(cosine_similarity([pv], [tv])[0][0])

eval_df = pd.read_csv(EVAL_CSV)
required_cols = ["question", "ground_truth"]
for c in required_cols:
    if c not in eval_df.columns:
        raise ValueError(f"Missing required column in evaluate.csv: {c}")

if "vbpl_extracted" not in eval_df.columns:
    eval_df["vbpl_extracted"] = ""

if "row_index" not in eval_df.columns:
    eval_df = eval_df.reset_index().rename(columns={"index": "row_index"})

if EVAL_LIMIT is not None:
    eval_df = eval_df.head(EVAL_LIMIT).copy()

sampled_df = eval_df.copy()
sheet = "evaluate.csv"
sample_stats = {
    "total": len(eval_df),
    "eligible": len(eval_df),
    "excluded_missing": 0,
    "excluded_empty_vbpl": 0,
}

results: list[dict[str, Any]] = []
for _, row in tqdm(sampled_df.iterrows(), total=len(sampled_df), desc="Evaluating"):
    question = fix_mojibake(str(row["question"])).strip()
    ground_truth = fix_mojibake(str(row["ground_truth"])).strip()
    vbpl = fix_mojibake(str(row.get("vbpl_extracted", ""))).strip()

    try:
        pred, final_chunks = rag_answer_for_question(question)
        err = ""
    except Exception as e:
        pred = ""
        final_chunks = []
        err = str(e)

    metrics = {
        "Cosine_Similarity": cosine_similarity_score(pred, ground_truth),
        "Jaccard_Similarity": jaccard_similarity(pred, ground_truth),
        "Token_Overlap": token_overlap_score(pred, ground_truth),
        "BLEU_Score": bleu_score_simple(pred, ground_truth),
        "ROUGE_L": rouge_l_score(pred, ground_truth),
    }

    results.append(
        {
            "row_index": int(row["row_index"]),
            "question": question,
            "ground_truth": ground_truth,
            "vbpl_extracted": vbpl,
            "prediction": pred,
            "retrieved_docs": "|".join(
                str((ch.get("payload", {}) or {}).get("document_number") or "")
                for ch in final_chunks
            ),
            "error": err,
            **metrics,
        }
    )

results_df = pd.DataFrame(results)
mean_metrics = {
    k: float(results_df[k].mean())
    for k in ["Cosine_Similarity", "Jaccard_Similarity", "Token_Overlap", "BLEU_Score", "ROUGE_L"]
}

out_dir = OUTPUT_DIR / RUN_NAME
out_dir.mkdir(parents=True, exist_ok=True)
results_csv = out_dir / "rag_eval_results.csv"
sampled_csv = out_dir / "sampled_questions.csv"
summary_json = out_dir / "summary.json"

results_df.to_csv(results_csv, index=False, encoding="utf-8-sig")
sampled_df.to_csv(sampled_csv, index=False, encoding="utf-8-sig")

summary = {
    "sheet": sheet,
    "sample_stats": sample_stats,
    "n": len(sampled_df),
    "seed": None,
    "mean_metrics": mean_metrics,
    "config": {
        "bm25_backend": "local_rank_bm25",
        "faiss_index_dir": str(FAISS_INDEX_DIR),
        "chunks_jsonl": str(CHUNKS_JSONL),
        "eval_csv": str(EVAL_CSV),
        "eval_limit": EVAL_LIMIT,
        "doc_focus_mode": DOC_FOCUS_MODE,
        "doc_focus_max_primary_docs": DOC_FOCUS_MAX_PRIMARY_DOCS,
        "doc_focus_primary_margin": DOC_FOCUS_PRIMARY_MARGIN,
        "doc_focus_primary_doc_quota": DOC_FOCUS_PRIMARY_DOC_QUOTA,
        "doc_focus_other_doc_quota": DOC_FOCUS_OTHER_DOC_QUOTA,
        "llm_backend": LLM_BACKEND,
        "llm_base_url": LLM_BASE_URL,
        "llm_model": LLM_MODEL,
    },
    "outputs": {
        "results_csv": str(results_csv),
        "sampled_csv": str(sampled_csv),
        "summary_json": str(summary_json),
    },
}
summary_json.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("=== FIXED EVAL SUMMARY ===")
print(f"sheet={sheet} | n={len(sampled_df)}")
for k, v in mean_metrics.items():
    print(f"{k}: {v:.4f}")
print(f"saved: {results_csv}")


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

Evaluating:  43%|████▎     | 43/100 [27:28<32:40, 34.39s/it]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Evaluating:  73%|███████▎  | 73/100 [46:19<19:41, 43.78s/it]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Evaluating:  86%|████████▌ | 86/100 [54:19<09:05, 38.94s/it]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Evaluating: 100%|██████████| 100/100 [1:01:41<00:00, 37.01s/it]


=== FIXED EVAL SUMMARY ===
sheet=evaluate.csv | n=100
Cosine_Similarity: 0.9632
Jaccard_Similarity: 0.5479
Token_Overlap: 0.7036
BLEU_Score: 0.4981
ROUGE_L: 0.4532
saved: /content/drive/MyDrive/LLM_RAG_PROJECT_SECOND/analysis/eval_runs/eval150_fixed_base_colab_local_bm25/rag_eval_results.csv
